# METHOD V2 STAGE 2 — ONE CLICK RUN

1. Open in Colab
2. Select **T4 GPU** (Runtime → Change runtime type → T4 GPU)
3. **Runtime → Run all**

Then wait. Nothing else to do — no folders to create, no datasets to fetch by hand,
no cells to edit.

Expect roughly **1h15m–2h** on a T4. Everything expensive is written to Drive and
reused, so pressing Run all again after a disconnect resumes rather than restarts.

The science is frozen. This notebook only executes it: the crop, DINOv2 ViT-B/14,
P2, NMS IoU 0.60, the REF-T1 cap and manifest, D/R1/R2/R3/C and every GO/NO-GO
threshold come from `docs/method_v2_stage2_protocol_2026-09-02.md` and are asserted,
not chosen here.

In [ ]:
#@title [1/10] Mounting Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib
DRIVE   = "/content/drive/MyDrive/OWL"
FEATURES = f"{DRIVE}/features"
pathlib.Path(FEATURES).mkdir(parents=True, exist_ok=True)

_probe = pathlib.Path(FEATURES) / ".write_probe"
_probe.write_text("ok"); assert _probe.read_text() == "ok"; _probe.unlink()
print(f"[1/10] Drive mounted; {FEATURES} is writable")

In [ ]:
#@title [2/10] Preparing repository (pinned)
import subprocess, sys

COMMIT = "8ca4ebaf348b366505782461b4f19599eeaeef4f"          # pinned; never floats to main
REPO   = "/content/owod-active"

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], check=True, **kw)

if not os.path.exists(REPO):
    run(["git", "clone", "-q", "https://github.com/gubiczam/owod-active.git", REPO])
run(["git", "-C", REPO, "fetch", "-q", "--all"])
run(["git", "-C", REPO, "checkout", "-q", COMMIT])
SHA = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
assert SHA == COMMIT, (SHA, COMMIT)
sys.path.insert(0, REPO)
print(f"[2/10] repository pinned at {SHA}")

In [ ]:
#@title [3/10] Installing dependencies
run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO])
run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "scipy"])

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
print("[3/10] GPU:", torch.cuda.get_device_name(0))

## [4/10] Materialising OWOD inputs

A fresh Colab `/content` is empty, so this fetches what Stage 2 opens. Reading the
tools rather than guessing: all three exporters read exactly one thing from the data
root — `JPEGImages/<image_id>.jpg`. No `Annotations/`, no `ImageSets/`, no PROB.
Boxes come from artefacts already committed in the repository.

The frozen REF-T1 manifest is asserted **first**, so a run that would produce a
different reference stops in seconds rather than after half an hour of downloading.
Resumable: valid JPEGs are validated and reused, never refetched or overwritten.

In [ ]:
#@title [4/10] Materialising OWOD inputs (~16,374 images, resumable)
DATA_ROOT = "/content/data/OWOD"

run([sys.executable, f"{REPO}/tools/bootstrap_stage2_data.py",
     "--data-root", DATA_ROOT,
     "--image-list-out", "/content/stage2_images.txt",
     "--workers", "32"])
print("[4/10] OWOD inputs ready")

In [ ]:
#@title [5/10] Reproducing the frozen REF-T1 manifest
from owl import reference_t1 as _ref
from tools.bootstrap_stage2_data import (
    EXPECTED_REF_T1_IMAGES, EXPECTED_REF_T1_MANIFEST, EXPECTED_REF_T1_OBJECTS,
)

_selection = _ref.select_balanced(
    _ref.enumerate_objects(), per_class_cap=_ref.PRIMARY_REF_T1_CAP_PER_CLASS)
_summary = _selection.summary()
_manifest = _selection.provenance["manifest_sha256"]

assert _summary["objects"] == EXPECTED_REF_T1_OBJECTS == 19000, _summary
assert _summary["images"] == EXPECTED_REF_T1_IMAGES == 14901, _summary
assert _summary["balanced"] is True, _summary
assert _manifest == EXPECTED_REF_T1_MANIFEST, _manifest

print(f"[5/10] REF-T1 frozen identity confirmed")
print(f"        objects  : {_summary['objects']:,} "
      f"({_summary['min_per_class']}/class x 19, exactly balanced)")
print(f"        images   : {_summary['images']:,}")
print(f"        manifest : {_manifest}")

## [6/10] Base DINOv2 export

The 80,000-proposal base export from the completed representation experiment is
reused from Drive if it is there and passes its own frozen validation gate.

If it is absent, it is **recreated automatically** rather than stopping the run.
That is scientifically identical, not a workaround: `export_dinov2_features.py` is
deterministic, uses the same frozen crop and backbone, and gates itself on
reproducing the committed pool. Recreating it changes nothing about the protocol.

In [ ]:
#@title [6/10] Base DINOv2 export (reuse from Drive, or recreate)
BASE_EXPORT = f"{FEATURES}/dinov2_vitb14_method_v2_v1.npz"

from owl import semantic_features as _sf

_rows = _sf.pool_rows(_sf.POOL)
_reusable = False
if os.path.exists(BASE_EXPORT):
    try:
        _report = _sf.validate(_sf.read(BASE_EXPORT), _rows)
        print(f"[6/10] reusing the existing base export: {_report}")
        _reusable = True
    except Exception as _error:
        print(f"[6/10] the existing base export failed validation: {_error}")
        print("        recreating it deterministically")

if not _reusable:
    run([sys.executable, f"{REPO}/tools/export_dinov2_features.py",
         "--data-root", DATA_ROOT, "--out", BASE_EXPORT,
         "--smoke-images", "4"])
    run([sys.executable, f"{REPO}/tools/export_dinov2_features.py",
         "--data-root", DATA_ROOT, "--out", BASE_EXPORT,
         "--batch-size", "128"])
    print(f"[6/10] base export written to {BASE_EXPORT}")

In [ ]:
#@title [7/10] REF-T1 smoke export (writes nothing)
REF_T1_EXPORT = f"{FEATURES}/ref_t1_dinov2_vitb14_cap1000_v1.npz"

if os.path.exists(REF_T1_EXPORT):
    print(f"[7/10] {REF_T1_EXPORT} already exists; skipping the smoke test")
else:
    run([sys.executable, f"{REPO}/tools/export_ref_t1_features.py",
         "--data-root", DATA_ROOT, "--out", REF_T1_EXPORT,
         "--per-class-cap", "1000", "--smoke-images", "4"])
    print("[7/10] REF-T1 smoke passed")

In [ ]:
#@title [8/10] Full REF-T1 export (19,000 references)
run([sys.executable, f"{REPO}/tools/export_ref_t1_features.py",
     "--data-root", DATA_ROOT, "--out", REF_T1_EXPORT,
     "--per-class-cap", "1000", "--batch-size", "128"])
print(f"[8/10] {REF_T1_EXPORT}")

In [ ]:
#@title [9/10] Consistency views: smoke, then full export (P2 only)
VIEWS_EXPORT = f"{FEATURES}/dinov2_vitb14_stage2_views_v1.npz"

if os.path.exists(VIEWS_EXPORT):
    print(f"[9/10] {VIEWS_EXPORT} already exists; skipping the smoke test")
else:
    run([sys.executable, f"{REPO}/tools/export_dinov2_consistency_views.py",
         "--data-root", DATA_ROOT, "--out", VIEWS_EXPORT, "--smoke-images", "4"])
    print("        consistency-view smoke passed")

run([sys.executable, f"{REPO}/tools/export_dinov2_consistency_views.py",
     "--data-root", DATA_ROOT, "--out", VIEWS_EXPORT, "--batch-size", "128"])
print(f"[9/10] {VIEWS_EXPORT}")

## [10/10] The frozen Stage-2 diagnostic

Run once, with all three exports. Every threshold and every definition is already
frozen in the repository; this cell chooses nothing.

In [ ]:
#@title [10/10] Stage-2 diagnostic (frozen)
for _path in (BASE_EXPORT, REF_T1_EXPORT, VIEWS_EXPORT):
    assert os.path.exists(_path), f"missing required export: {_path}"

run([sys.executable, f"{REPO}/tools/diagnose_method_v2_stage2.py",
     "--export", BASE_EXPORT,
     "--ref-t1", REF_T1_EXPORT,
     "--views",  VIEWS_EXPORT,
     "--with-pseudo-reference"],
    cwd=REPO)

In [ ]:
#@title Final summary — results are copied to Drive
import json, shutil

# thresholds are read from the frozen module, never restated here
from owl.method_v2_stage2 import (
    C_GO_UNKNOWN_VS_BACKGROUND_AUC, D_GO_UNKNOWN_VS_KNOWN_AUC,
)

_results = pathlib.Path(REPO) / "data" / "results"
_dest = pathlib.Path(DRIVE) / "results" / "method_v2_stage2"
_dest.mkdir(parents=True, exist_ok=True)

_copied = []
for _pattern in ("method_v2_stage2_*.csv", "method_v2_stage2_*.json"):
    for _path in sorted(_results.glob(_pattern)):
        shutil.copy2(_path, _dest / _path.name)
        _copied.append(_dest / _path.name)

_summary = json.loads((_results / "method_v2_stage2_summary.json").read_text())
_d, _r, _c = _summary["D"]["go"], _summary["R"]["go"], _summary["C"]["go"]

print("=" * 70)
print("METHOD V2 STAGE 2 — FINAL RESULT")
print("=" * 70)
print(f"  D_{'GO' if _d else 'NO_GO'}")
print(f"  R_{'GO' if _r else 'NO_GO'}")
print(f"  C_{'GO' if _c else 'NO_GO'}")
print()
print(f"  METHOD_V2_ALLOWED_LADDER = {_summary['ladder']}")
print()
print(f"  D  unknown-vs-known AUC          : {_summary['D']['unknown_vs_known_auc']:.4f}  "
      f"(>= {D_GO_UNKNOWN_VS_KNOWN_AUC})")
print(f"  R  passing definitions           : {_summary['R']['passing_definitions'] or 'none'}")
print(f"  C  unknown-vs-background AUC     : "
      f"{_summary['C'].get('unknown_vs_background_auc', float('nan')):.4f}  "
      f"(>= {C_GO_UNKNOWN_VS_BACKGROUND_AUC})")
print(f"  P2 population                    : {_summary['p2']['rows']:,} rows, "
      f"background {_summary['p2']['background_share']:.3f}")
print(f"  REF-T1 reference vectors         : {_summary['reference_vectors']:,}")
print()
print("  results (also on Drive):")
for _path in _copied:
    print(f"    {_path}")
print()
print("  features on Drive:")
for _path in (BASE_EXPORT, REF_T1_EXPORT, VIEWS_EXPORT):
    print(f"    {_path}")
print("=" * 70)